# Predicting 30-Day Hospital Readmission Among Patients With Diabetes

**Portfolio version**

This project predicts whether a patient with diabetes will be readmitted to the hospital within 30 days of discharge.

The analysis uses a large multi-hospital dataset and compares three classification approaches:

1. Logistic regression
2. L1-regularized logistic regression (LASSO-style)
3. Classification tree

Because 30-day readmission is relatively uncommon, model performance is evaluated primarily with **ROC-AUC** rather than accuracy alone. The project also evaluates a **cost-sensitive decision threshold** under the assumption that missing a true readmission is twice as costly as incorrectly flagging a patient who is not readmitted.

> **Note on cleanup:** This portfolio notebook contains only the Mini Project section from the original course notebook. Repeated imports, assignment material, and unrelated homework sections were removed. The decision-tree evaluation is also kept on the same unscaled feature representation used to train the tree; this corrects an inconsistency in the original working notebook, so tree metrics may differ slightly from values recorded in the submitted report.

## 1. Setup and Imports

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

DATA_PATH = Path("../data/readmission.csv")
RANDOM_STATE = 0

## 2. Load the Data

The dataset contains hospital admissions for patients with diabetes across 130 U.S. hospitals between 1999 and 2008.

The outcome variable records whether the patient was readmitted:

- `<30` — readmitted within 30 days
- `>30` — readmitted after 30 days
- `NO` — not readmitted

For modeling, the outcome is converted to a binary target where **1 = readmitted within 30 days** and **0 = otherwise**.

In [ ]:
readmission = pd.read_csv(DATA_PATH)

print(f"Dataset shape: {readmission.shape}")
readmission.head()

## 3. Exploratory Data Analysis

The submitted analysis found that approximately 11% of encounters resulted in a readmission within 30 days, creating a substantially imbalanced classification problem.

In [ ]:
readmission_counts = (
    readmission["readmitted"]
    .value_counts()
    .reindex(["NO", "<30", ">30"])
)

plt.figure(figsize=(7, 5))
plt.bar(readmission_counts.index, readmission_counts.values)
plt.title("Distribution of Readmission Categories")
plt.xlabel("Readmission Category")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

In [ ]:
age_counts = readmission["age_mod"].value_counts()

# Use an intuitive age-group order when all expected categories are present.
preferred_order = ["0-19", "20-59", "60-79", "80+"]
age_order = [x for x in preferred_order if x in age_counts.index]

plt.figure(figsize=(7, 5))
plt.bar(age_order, age_counts.reindex(age_order).values)
plt.title("Age Group Distribution")
plt.xlabel("Age Group")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

In [ ]:
numeric_cols = [
    "time_in_hospital",
    "num_lab_procedures",
    "num_procedures",
    "num_medications",
    "number_outpatient",
    "number_emergency",
    "number_inpatient",
    "number_diagnoses",
]

readmission[numeric_cols].describe().T

### Data-quality checks

The original project identified missing values in `max_glu_serum` and `A1Cresult` and treated missing observations as `"None"` for modeling. Identifier columns are excluded from prediction.

In [ ]:
for col in ["max_glu_serum", "A1Cresult"]:
    readmission[col] = readmission[col].fillna("None")

print("Missing values in selected lab-result fields:")
print(readmission[["max_glu_serum", "A1Cresult"]].isna().sum())

## 4. Prepare the Modeling Data

Categorical variables are one-hot encoded. The data are then split into:

- **60% training**
- **20% testing**
- **20% validation**

The test set is used to compare candidate models. The validation set is reserved for the final LASSO evaluation and threshold analysis.

In [ ]:
df = readmission.copy()

# Binary target: 1 = readmitted within 30 days, 0 = otherwise.
y = (df["readmitted"] == "<30").astype(int)

# Remove identifiers and the original outcome from predictors.
X_raw = df.drop(columns=["encounter_id", "patient_nbr", "readmitted"])

categorical_cols = [
    "race",
    "gender",
    "max_glu_serum",
    "A1Cresult",
    "metformin",
    "glimepiride",
    "glipizide",
    "glyburide",
    "pioglitazone",
    "rosiglitazone",
    "insulin",
    "change",
    "diabetesMed",
    "disch_disp_modified",
    "adm_src_mod",
    "adm_typ_mod",
    "age_mod",
    "diag1_mod",
    "diag2_mod",
    "diag3_mod",
]

X = pd.get_dummies(
    X_raw,
    columns=categorical_cols,
    drop_first=True,
)

print(f"Encoded predictor matrix shape: {X.shape}")
print(f"30-day readmission rate: {y.mean():.3%}")

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.40,
    random_state=RANDOM_STATE,
    stratify=y,
)

X_test, X_val, y_test, y_val = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=RANDOM_STATE,
    stratify=y_temp,
)

print(f"Training observations:   {len(X_train):,}")
print(f"Testing observations:    {len(X_test):,}")
print(f"Validation observations: {len(X_val):,}")

## 5. Scale Features for Regression Models

Logistic regression models are fit on standardized features. The scaler is fit only on the training data and then applied to the test and validation sets.

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_val_scaled = scaler.transform(X_val)

## 6. Fit Candidate Models

### 6.1 Logistic Regression

Class weights are balanced to account for the relatively small proportion of 30-day readmissions.

In [ ]:
log_reg = LogisticRegression(
    penalty="l2",
    solver="lbfgs",
    max_iter=1000,
    class_weight="balanced",
)

log_reg.fit(X_train_scaled, y_train)

### 6.2 LASSO Logistic Regression

An L1 penalty shrinks weaker coefficients toward zero, providing a form of feature selection in the high-dimensional dummy-encoded predictor set.

In [ ]:
lasso = LogisticRegression(
    penalty="l1",
    solver="saga",
    C=0.5,
    max_iter=2000,
    class_weight="balanced",
    random_state=RANDOM_STATE,
)

lasso.fit(X_train_scaled, y_train)

### 6.3 Classification Tree

The tree is constrained to reduce overfitting and is trained on the original unscaled predictors.

In [ ]:
tree = DecisionTreeClassifier(
    max_depth=5,
    min_samples_split=50,
    min_samples_leaf=25,
    class_weight="balanced",
    random_state=RANDOM_STATE,
)

tree.fit(X_train, y_train)

## 7. Compare Models on the Test Set

ROC-AUC is used as the primary comparison metric because it evaluates how well each model ranks readmitted versus non-readmitted patients without depending on a single classification threshold.

In [ ]:
# Logistic regression
y_test_prob_log = log_reg.predict_proba(X_test_scaled)[:, 1]
auc_log = roc_auc_score(y_test, y_test_prob_log)

# LASSO logistic regression
y_test_prob_lasso = lasso.predict_proba(X_test_scaled)[:, 1]
auc_lasso = roc_auc_score(y_test, y_test_prob_lasso)

# Decision tree -- evaluated on the same unscaled representation used for training.
y_test_prob_tree = tree.predict_proba(X_test)[:, 1]
auc_tree = roc_auc_score(y_test, y_test_prob_tree)

results_test = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "LASSO Logistic Regression",
        "Decision Tree",
    ],
    "Test AUC": [
        auc_log,
        auc_lasso,
        auc_tree,
    ],
}).sort_values("Test AUC", ascending=False)

results_test

In [ ]:
plt.figure(figsize=(8, 5))
bars = plt.barh(
    results_test["Model"],
    results_test["Test AUC"],
)

plt.xlabel("Test ROC-AUC")
plt.title("Model Comparison")
plt.xlim(0.5, max(0.7, results_test["Test AUC"].max() + 0.03))

for bar in bars:
    value = bar.get_width()
    plt.text(
        value + 0.003,
        bar.get_y() + bar.get_height() / 2,
        f"{value:.3f}",
        va="center",
    )

plt.tight_layout()
plt.show()

The submitted report selected the LASSO logistic regression model as the final model because it achieved the strongest test AUC while also providing feature-selection and interpretability benefits.

## 8. Identify Influential LASSO Predictors

The largest non-zero coefficients indicate the features most strongly associated with the model's predicted probability of 30-day readmission, relative to each feature's encoded reference category.

In [ ]:
coef_series = pd.Series(
    lasso.coef_[0],
    index=X.columns,
    name="coefficient",
)

nonzero = coef_series[coef_series != 0]
top_predictors = (
    nonzero
    .reindex(nonzero.abs().sort_values(ascending=False).index)
    .head(20)
)

top_predictors.to_frame()

In [ ]:
plot_values = top_predictors.sort_values()

plt.figure(figsize=(10, 7))
plt.barh(
    plot_values.index,
    plot_values.values,
)
plt.axvline(0, linewidth=1)
plt.title("Top 20 LASSO Predictors by Absolute Coefficient")
plt.xlabel("Standardized Logistic Regression Coefficient")
plt.ylabel("Predictor")
plt.tight_layout()
plt.show()

In the submitted analysis, influential predictors included prior inpatient utilization, patient age group, discharge disposition, diagnosis categories, and medication patterns. Prior inpatient visits were among the strongest positive predictors of 30-day readmission risk.

## 9. Final Validation Evaluation

The selected LASSO model is evaluated once on the held-out validation set using the standard 0.50 probability threshold.

In [ ]:
y_val_prob_lasso = lasso.predict_proba(X_val_scaled)[:, 1]
y_val_pred_050 = (y_val_prob_lasso >= 0.50).astype(int)

val_auc_lasso = roc_auc_score(y_val, y_val_prob_lasso)

print("LASSO validation performance at threshold = 0.50")
print(classification_report(y_val, y_val_pred_050))
print(f"Validation ROC-AUC: {val_auc_lasso:.4f}")

The submitted report found a validation AUC of approximately **0.64**. At the 0.50 threshold, recall for readmission was roughly **0.53**, meaning the model detected just over half of true 30-day readmissions.

## 10. Cost-Sensitive Threshold Analysis

Suppose a false negative — failing to flag a patient who is actually readmitted — is assigned twice the cost of a false positive.

Under a simple theoretical decision rule:

- False-negative cost = 2
- False-positive cost = 1

the implied probability threshold is:

**1 / (1 + 2) = 1/3 ≈ 0.33**

We compare the standard 0.50 threshold with this cost-sensitive threshold on the validation data.

In [ ]:
def evaluate_threshold_cost(
    y_true,
    probabilities,
    threshold,
    false_negative_cost=2,
    false_positive_cost=1,
):
    predictions = (probabilities >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        predictions,
    ).ravel()

    total_cost = (
        false_negative_cost * fn
        + false_positive_cost * fp
    )

    return {
        "Threshold": threshold,
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp,
        "Recall": tp / (tp + fn),
        "Total Cost": total_cost,
    }

threshold_results = pd.DataFrame([
    evaluate_threshold_cost(
        y_val,
        y_val_prob_lasso,
        threshold=0.50,
    ),
    evaluate_threshold_cost(
        y_val,
        y_val_prob_lasso,
        threshold=1/3,
    ),
])

threshold_results

In [ ]:
plt.figure(figsize=(7, 5))
bars = plt.bar(
    threshold_results["Threshold"].astype(str),
    threshold_results["Total Cost"],
)

plt.xlabel("Classification Threshold")
plt.ylabel("Weighted Misclassification Cost")
plt.title("Cost Comparison by Classification Threshold")

for bar in bars:
    value = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        value,
        f"{value:,.0f}",
        ha="center",
        va="bottom",
    )

plt.tight_layout()
plt.show()

### Threshold trade-off

The submitted analysis found that lowering the threshold from 0.50 to approximately 0.33 greatly increased recall for true readmissions, but also produced many more false positives.

Despite the assumed 2:1 false-negative-to-false-positive cost ratio, the lower threshold produced a **higher observed total cost** on the validation set. This illustrates why a theoretically motivated threshold should still be evaluated empirically on held-out data.

## 11. Key Findings

- The dataset contains more than 100,000 hospital encounters and is strongly imbalanced, with only about 11% resulting in a 30-day readmission.
- Logistic regression and LASSO produced similar test discrimination, with LASSO selected as the final model because it also supports feature selection.
- Prior inpatient utilization and age were among the most influential predictors in the submitted LASSO analysis.
- Overall discrimination was modest, with ROC-AUC values around 0.65.
- Lowering the classification threshold substantially increased readmission recall but also generated many false positives.
- In the submitted validation analysis, the standard 0.50 threshold produced a lower weighted misclassification cost than the theoretically derived 0.33 threshold.

## 12. Limitations

- The data are highly imbalanced.
- Predictive performance is modest rather than strong.
- The dataset covers hospital encounters from 1999–2008 and may not reflect current clinical practice.
- Important information such as social determinants of health and unstructured clinician notes is unavailable.
- The analysis is predictive and observational; identified relationships should not be interpreted as causal.
- The assumed 2:1 cost ratio is simplified and may not reflect actual hospital financial or clinical consequences.